# 🐔 Freeway Atari DQN Agent
**Deep Q-Network with CNN, 100k Replay Buffer & Live Visualization**

Trains a DQN agent on the real **ALE/Freeway-v5** Atari environment using pixel observations.

| Component | Details |
|---|---|
| Environment | `ALE/Freeway-v5` (Atari via Gymnasium + ALE) |
| Network | CNN: 3 conv layers → 512-dim FC → Q-values |
| Input | 84×84 grayscale frame stack (4 frames) |
| Optimizer | Adam (lr=0.0001) |
| Replay buffer | 100,000 transitions |
| Reward | Native Atari reward (1 per crossing) |
| Visualization | Live reward/loss/Q-value charts |

## 1 · Install & Import Dependencies

In [ ]:
# Install dependencies (run once)
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gymnasium[atari,accept-rom-license]',
    'ale-py',
    'numpy', 'torch', 'matplotlib', 'IPython',
    'opencv-python-headless',
], check=True)

print('Dependencies ready.')

In [ ]:
import random
import math
import collections
from typing import List, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor']   = '#161b22'
matplotlib.rcParams['text.color']       = '#c9d1d9'
matplotlib.rcParams['axes.labelcolor']  = '#c9d1d9'
matplotlib.rcParams['xtick.color']      = '#8b949e'
matplotlib.rcParams['ytick.color']      = '#8b949e'
matplotlib.rcParams['axes.edgecolor']   = '#30363d'
matplotlib.rcParams['grid.color']       = '#21262d'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2 · Environment Wrappers

We wrap `ALE/Freeway-v5` with standard Atari preprocessing:
- **Grayscale** conversion
- **84×84** resize
- **Frame stacking** (4 frames) so the network can infer motion
- **NoopReset** and **MaxAndSkip** for standard Atari training

In [ ]:
class NoopResetEnv(gym.Wrapper):
    """Execute a random number of no-ops on reset to introduce stochasticity."""
    def __init__(self, env, noop_max=30):
        super().__init__(env)
        self.noop_max = noop_max
        self.noop_action = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        noops = random.randint(1, self.noop_max)
        for _ in range(noops):
            obs, _, terminated, truncated, info = self.env.step(self.noop_action)
            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
        return obs, info


class MaxAndSkipEnv(gym.Wrapper):
    """Return max over the last 2 frames; repeat action for `skip` steps."""
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip
        self._obs_buffer = collections.deque(maxlen=2)

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if terminated or truncated:
                break
        # Element-wise max over last 2 frames to remove flicker
        max_frame = np.max(np.stack(list(self._obs_buffer), axis=0), axis=0)
        return max_frame, total_reward, terminated, truncated, info


class GrayscaleResizeObs(gym.ObservationWrapper):
    """Convert RGB frame to 84×84 grayscale, output shape (1, 84, 84)."""
    def __init__(self, env, width=84, height=84):
        super().__init__(env)
        self.width  = width
        self.height = height
        self.observation_space = gym.spaces.Box(
            low=0, high=255,
            shape=(1, height, width),
            dtype=np.uint8,
        )

    def observation(self, obs):
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (self.width, self.height),
                             interpolation=cv2.INTER_AREA)
        return resized[np.newaxis, :, :]   # (1, 84, 84)


class FrameStackEnv(gym.Wrapper):
    """Stack the last `k` frames along axis 0 → shape (k, 84, 84)."""
    def __init__(self, env, k=4):
        super().__init__(env)
        self.k = k
        self._frames = collections.deque(maxlen=k)
        shp = env.observation_space.shape   # (1, 84, 84)
        self.observation_space = gym.spaces.Box(
            low=0, high=255,
            shape=(k * shp[0], shp[1], shp[2]),
            dtype=np.uint8,
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.k):
            self._frames.append(obs)
        return self._get_obs(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self._frames.append(obs)
        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self):
        return np.concatenate(list(self._frames), axis=0)  # (4, 84, 84)


def make_env(render_mode=None, seed=42):
    """Build the fully-wrapped Freeway environment."""
    env = gym.make('ALE/Freeway-v5',
                   render_mode=render_mode,
                   frameskip=1,          # we handle skip ourselves
                   repeat_action_probability=0.0)
    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = GrayscaleResizeObs(env)
    env = FrameStackEnv(env, k=4)
    env.reset(seed=seed)
    return env


# Quick sanity check
env_test = make_env(seed=0)
obs, _ = env_test.reset()
print(f'Observation shape: {obs.shape}, dtype: {obs.dtype}')
print(f'Action space:      {env_test.action_space}')
obs2, r, term, trunc, info = env_test.step(1)
print(f'After UP → reward={r}, terminated={term}')
env_test.close()

## 3 · CNN Q-Network

Classic DQN CNN architecture (Mnih et al. 2015):

```
Input (4, 84, 84)
  → Conv(32, 8×8, stride 4) → ReLU
  → Conv(64, 4×4, stride 2) → ReLU
  → Conv(64, 3×3, stride 1) → ReLU
  → Flatten → Linear(512) → ReLU
  → Linear(n_actions)   ← Q-values
```

In [ ]:
class DQN(nn.Module):
    """
    CNN Q-network for pixel-based Atari observations.

    Input:  float tensor of shape (B, 4, 84, 84), values in [0, 1]
    Output: Q-value tensor of shape (B, n_actions)
    """

    def __init__(self, in_channels: int = 4, n_actions: int = 3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),  # → (32, 20, 20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),           # → (64,  9,  9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),           # → (64,  7,  7)
            nn.ReLU(),
        )
        conv_out = self._conv_out_size(in_channels)
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )
        self._init_weights()

    def _conv_out_size(self, in_channels: int) -> int:
        dummy = torch.zeros(1, in_channels, 84, 84)
        return int(np.prod(self.conv(dummy).shape[1:]))

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 4, 84, 84) uint8 → normalise to float [0, 1]
        if x.dtype == torch.uint8:
            x = x.float() / 255.0
        return self.fc(self.conv(x).flatten(1))


# Preview architecture
_env = make_env(seed=0)
N_ACTIONS = _env.action_space.n
_env.close()

net = DQN(in_channels=4, n_actions=N_ACTIONS).to(DEVICE)
print(net)
total_params = sum(p.numel() for p in net.parameters())
print(f'\nTotal parameters: {total_params:,}')
print(f'Action space size: {N_ACTIONS}')

## 4 · Replay Buffer (100,000 transitions)

In [ ]:
Transition = collections.namedtuple(
    'Transition', ['state', 'action', 'reward', 'next_state', 'done'])


class ReplayBuffer:
    """Circular replay buffer capped at `capacity` transitions.
    
    Stores states as uint8 (4×84×84) to keep memory footprint low.
    """

    def __init__(self, capacity: int = 100_000):
        self.capacity = capacity
        self.buffer: collections.deque = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(
            np.array(state,      dtype=np.uint8),
            int(action),
            float(reward),
            np.array(next_state, dtype=np.uint8),
            bool(done),
        ))

    def sample(self, batch_size: int) -> Tuple:
        batch = random.sample(self.buffer, batch_size)
        states      = torch.tensor(np.stack([t.state      for t in batch]),
                                   dtype=torch.uint8, device=DEVICE)
        actions     = torch.tensor([t.action for t in batch],
                                   dtype=torch.long, device=DEVICE)
        rewards     = torch.tensor([t.reward for t in batch],
                                   dtype=torch.float32, device=DEVICE)
        next_states = torch.tensor(np.stack([t.next_state for t in batch]),
                                   dtype=torch.uint8, device=DEVICE)
        dones       = torch.tensor([t.done for t in batch],
                                   dtype=torch.float32, device=DEVICE)
        return states, actions, rewards, next_states, dones

    def __len__(self) -> int:
        return len(self.buffer)

    @property
    def pct_full(self) -> float:
        return len(self.buffer) / self.capacity * 100


buffer = ReplayBuffer(capacity=100_000)
print(f'Buffer created — capacity: {buffer.capacity:,} transitions')

# Estimate memory footprint
bytes_per_transition = 2 * (4 * 84 * 84)  # uint8, state + next_state
total_mb = bytes_per_transition * buffer.capacity / 1e6
print(f'Estimated max memory: ~{total_mb:.0f} MB')

## 5 · DQN Agent

In [ ]:
class DQNAgent:
    """
    DQN agent with:
      - ε-greedy exploration with exponential decay
      - target network (hard update every TARGET_UPDATE steps)
      - Adam optimiser
      - Huber (smooth-L1) loss for stability
      - gradient clipping
    """

    LR            = 1e-4       # lower LR suits CNN + pixel inputs
    GAMMA         = 0.99       # longer horizon for Atari
    EPSILON_START = 1.0
    EPSILON_END   = 0.05
    EPSILON_DECAY = 0.9995
    BATCH_SIZE    = 32
    TARGET_UPDATE = 1000       # update target net every N training steps
    MIN_BUFFER    = 1000       # don't train until buffer has this many

    def __init__(self, n_actions: int = 3):
        self.n_actions   = n_actions
        self.epsilon     = self.EPSILON_START
        self.train_steps = 0
        self.losses: List[float] = []

        self.q_net  = DQN(in_channels=4, n_actions=n_actions).to(DEVICE)
        self.t_net  = DQN(in_channels=4, n_actions=n_actions).to(DEVICE)
        self.t_net.load_state_dict(self.q_net.state_dict())
        self.t_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=self.LR)

    # ── action selection ──────────────────────────────────────────────────
    def select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            s = torch.tensor(state[np.newaxis], dtype=torch.uint8,
                             device=DEVICE)
            return self.q_net(s).argmax(dim=1).item()

    def get_q_values(self, state: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            s = torch.tensor(state[np.newaxis], dtype=torch.uint8,
                             device=DEVICE)
            return self.q_net(s).cpu().numpy()[0]

    # ── training step ─────────────────────────────────────────────────────
    def train_step(self, replay: ReplayBuffer) -> Optional[float]:
        if len(replay) < self.MIN_BUFFER:
            return None

        states, actions, rewards, next_states, dones = replay.sample(self.BATCH_SIZE)

        # Current Q-values
        current_q = self.q_net(states).gather(
            1, actions.unsqueeze(1)).squeeze(1)

        # Target Q-values (Bellman equation)
        with torch.no_grad():
            max_next_q = self.t_net(next_states).max(dim=1)[0]
            target_q   = rewards + self.GAMMA * max_next_q * (1 - dones)

        loss = F.smooth_l1_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()

        self.train_steps += 1
        if self.train_steps % self.TARGET_UPDATE == 0:
            self.t_net.load_state_dict(self.q_net.state_dict())

        self.epsilon = max(self.EPSILON_END,
                           self.epsilon * self.EPSILON_DECAY)
        loss_val = loss.item()
        self.losses.append(loss_val)
        return loss_val


agent = DQNAgent(n_actions=N_ACTIONS)
print('Agent ready.')
print(f'  Q-network params: {sum(p.numel() for p in agent.q_net.parameters()):,}')

## 6 · Visualization Helpers

In [ ]:
def render_frame(ax, frame: np.ndarray, episode: int, ep_reward: float,
                 epsilon: float, ep_crossings: int):
    """Display the latest grayscale frame (channel 3 of the 4-stack)."""
    ax.clear()
    ax.imshow(frame[3], cmap='gray', vmin=0, vmax=255, aspect='equal')
    ax.axis('off')
    ax.set_title(
        f'Episode {episode}  |  Reward {ep_reward:.1f}  |  '
        f'ε={epsilon:.3f}  |  Crossings {ep_crossings}',
        color='#c9d1d9', fontsize=8, pad=4
    )


def render_charts(ax_rw, ax_loss, ax_q,
                  reward_history, loss_history, q_values,
                  buffer_pct: float):
    """Update the reward, loss, and Q-value charts."""
    # ── reward ─────────────────────────────────────────────────────────
    ax_rw.clear()
    if reward_history:
        window = reward_history[-150:]
        ax_rw.plot(window, color='#378ADD', lw=1.0, alpha=0.7,
                   label='Episode reward')
        if len(window) >= 10:
            roll = np.convolve(window, np.ones(10)/10, mode='valid')
            ax_rw.plot(range(9, len(window)), roll,
                       color='#1D9E75', lw=1.5, label='10-ep avg')
        ax_rw.axhline(0, color='#8b949e', lw=0.5, ls='--')
        ax_rw.legend(fontsize=7, loc='upper left')
    ax_rw.set_title(f'Reward  (buffer {buffer_pct:.1f}% full)',
                    color='#c9d1d9', fontsize=8)
    ax_rw.set_xlabel('Episode', fontsize=7)

    # ── loss ───────────────────────────────────────────────────────────
    ax_loss.clear()
    if loss_history:
        ax_loss.plot(loss_history[-300:], color='#D4537E', lw=0.8, alpha=0.8)
    ax_loss.set_title('Training loss (Huber)', color='#c9d1d9', fontsize=8)
    ax_loss.set_xlabel('Train step', fontsize=7)

    # ── Q-values ────────────────────────────────────────────────────────
    ax_q.clear()
    labels = ['NOOP', 'UP', 'DOWN']
    colors = ['#378ADD', '#1D9E75', '#D4537E']
    bars = ax_q.bar(labels, q_values[:3], color=colors, width=0.5)
    for bar, val in zip(bars, q_values[:3]):
        ax_q.text(bar.get_x() + bar.get_width()/2,
                  val + (0.02 if val >= 0 else -0.12),
                  f'{val:.2f}', ha='center', va='bottom',
                  fontsize=8, color='#c9d1d9')
    ax_q.axhline(0, color='#8b949e', lw=0.5)
    ax_q.set_title('Q-values (current state)', color='#c9d1d9', fontsize=8)


print('Visualization helpers defined.')

## 7 · Training Loop

> **Configuration** — adjust these before running.

**Note:** Atari pixel-based DQN takes significantly longer per episode than a
hand-coded sim. For meaningful learning, run ≥ 500 episodes.
GPU is strongly recommended for faster training.

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
NUM_EPISODES    = 500      # total training episodes
RENDER_EVERY    = 25       # update plots every N episodes
LOG_EVERY       = 50       # print summary every N episodes
BUFFER_CAPACITY = 100_000  # replay buffer size

# ── Re-initialise everything ─────────────────────────────────────────────────
env    = make_env(seed=0)
agent  = DQNAgent(n_actions=N_ACTIONS)
buffer = ReplayBuffer(capacity=BUFFER_CAPACITY)

reward_history: List[float] = []
total_crossings = 0

print(f'Training for {NUM_EPISODES} episodes, buffer capacity {BUFFER_CAPACITY:,}')

In [ ]:
%matplotlib inline

fig = plt.figure(figsize=(14, 7))
ax_game  = fig.add_subplot(1, 2, 1)
ax_rw    = fig.add_subplot(3, 2, 2)
ax_loss  = fig.add_subplot(3, 2, 4)
ax_q     = fig.add_subplot(3, 2, 6)
fig.tight_layout(pad=2.5)

for ep in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    ep_reward    = 0.0
    ep_crossings = 0
    prev_score   = 0
    terminated   = truncated = False

    while not (terminated or truncated):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        buffer.push(state, action, reward, next_state, done)
        agent.train_step(buffer)

        # Count crossings: the Atari score increments by 1 each time the
        # chicken reaches the top
        score = info.get('score', 0)
        if score > prev_score:
            ep_crossings  += score - prev_score
            total_crossings += score - prev_score
            prev_score = score

        state      = next_state
        ep_reward += reward

    reward_history.append(ep_reward)

    # ── render ─────────────────────────────────────────────────────────
    if ep % RENDER_EVERY == 0 or ep == 1:
        q_vals = agent.get_q_values(state)
        render_frame(ax_game, state, ep, ep_reward,
                     agent.epsilon, ep_crossings)
        render_charts(ax_rw, ax_loss, ax_q,
                      reward_history, agent.losses, q_vals,
                      buffer.pct_full)
        fig.suptitle(
            f'Freeway Atari DQN  —  Buffer {len(buffer):,}/{BUFFER_CAPACITY:,} '
            f'({buffer.pct_full:.1f}%)  |  Train steps {agent.train_steps:,}',
            color='#c9d1d9', fontsize=9, y=1.01
        )
        clear_output(wait=True)
        display(fig)

    # ── log ────────────────────────────────────────────────────────────
    if ep % LOG_EVERY == 0:
        avg50 = np.mean(reward_history[-50:])
        print(
            f'Ep {ep:4d} | reward {ep_reward:7.2f} | '
            f'avg50 {avg50:6.2f} | ε {agent.epsilon:.4f} | '
            f'buf {len(buffer):,} | crossings {total_crossings}'
        )

env.close()
print('\nTraining complete.')

## 8 · Post-Training Analysis

In [ ]:
fig2, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Reward curve ─────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(reward_history, color='#378ADD', lw=0.8, alpha=0.5, label='Episode reward')
window = 20
if len(reward_history) >= window:
    roll = np.convolve(reward_history, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(reward_history)), roll,
            color='#1D9E75', lw=2, label=f'{window}-ep rolling avg')
ax.axhline(0, color='#8b949e', lw=0.5, ls='--')
ax.set_title('Reward over training', fontsize=10)
ax.set_xlabel('Episode')
ax.legend(fontsize=8)

# ── Loss curve ───────────────────────────────────────────────────────────────
ax = axes[1]
if agent.losses:
    ax.plot(agent.losses, color='#D4537E', lw=0.6, alpha=0.6)
    roll_loss = np.convolve(agent.losses, np.ones(50)/50, mode='valid')
    ax.plot(range(49, len(agent.losses)), roll_loss,
            color='#EF9F27', lw=1.5, label='50-step avg')
    ax.legend(fontsize=8)
ax.set_title('Huber loss', fontsize=10)
ax.set_xlabel('Train step')

# ── ε decay ──────────────────────────────────────────────────────────────────
ax = axes[2]
eps_start = DQNAgent.EPSILON_START
eps_end   = DQNAgent.EPSILON_END
eps_decay = DQNAgent.EPSILON_DECAY
eps_curve = [max(eps_end, eps_start * eps_decay**i)
             for i in range(agent.train_steps + 1)]
ax.plot(eps_curve[::50], color='#BA7517', lw=1.5)
ax.axhline(eps_end, color='#8b949e', lw=0.5, ls='--', label=f'min ε={eps_end}')
ax.set_title('ε decay schedule', fontsize=10)
ax.set_xlabel('Train step (×50)')
ax.legend(fontsize=8)

fig2.suptitle('Post-Training Analysis', fontsize=12, y=1.02)
fig2.tight_layout()
plt.show()

print(f'Final ε:                  {agent.epsilon:.4f}')
print(f'Total crossings:          {total_crossings}')
print(f'Buffer fill:              {len(buffer):,} / {buffer.capacity:,} ({buffer.pct_full:.1f}%)')
print(f'Avg reward (last 50 eps): {np.mean(reward_history[-50:]):.2f}')

## 9 · Watch the Trained Agent Play

In [ ]:
EVAL_EPISODES = 5

eval_env  = make_env(seed=99)
fig3, ax3 = plt.subplots(figsize=(4, 6))

for ep_i in range(1, EVAL_EPISODES + 1):
    state, _  = eval_env.reset()
    terminated = truncated = False
    ep_r       = 0.0
    ep_cross   = 0
    prev_score = 0

    while not (terminated or truncated):
        # Greedy policy during evaluation
        with torch.no_grad():
            s = torch.tensor(state[np.newaxis], dtype=torch.uint8,
                             device=DEVICE)
            action = agent.q_net(s).argmax(dim=1).item()

        state, r, terminated, truncated, info = eval_env.step(action)
        ep_r += r

        score = info.get('score', 0)
        if score > prev_score:
            ep_cross += score - prev_score
            prev_score = score

        render_frame(ax3, state, ep_i, ep_r, agent.epsilon, ep_cross)
        clear_output(wait=True)
        display(fig3)

    print(f'Eval episode {ep_i}: reward={ep_r:.2f}, crossings={ep_cross}')

eval_env.close()
plt.close(fig3)
print('Evaluation complete.')

## 10 · Save & Load the Model

In [ ]:
import os

CHECKPOINT_PATH = 'freeway_atari_dqn_checkpoint.pt'

# ── Save ─────────────────────────────────────────────────────────────────────
torch.save({
    'q_net_state':     agent.q_net.state_dict(),
    't_net_state':     agent.t_net.state_dict(),
    'optimizer_state': agent.optimizer.state_dict(),
    'epsilon':         agent.epsilon,
    'train_steps':     agent.train_steps,
    'reward_history':  reward_history,
    'total_crossings': total_crossings,
    'n_actions':       N_ACTIONS,
}, CHECKPOINT_PATH)
print(f'Saved → {CHECKPOINT_PATH}  ({os.path.getsize(CHECKPOINT_PATH)/1024:.1f} KB)')

# ── Load example ─────────────────────────────────────────────────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
agent2 = DQNAgent(n_actions=checkpoint['n_actions'])
agent2.q_net.load_state_dict(checkpoint['q_net_state'])
agent2.t_net.load_state_dict(checkpoint['t_net_state'])
agent2.epsilon = checkpoint['epsilon']
print(f'Loaded agent — ε={agent2.epsilon:.4f}, train_steps={checkpoint["train_steps"]}')

---
### Architecture Summary

```
Input (4, 84, 84)  ← 4 stacked grayscale frames
  → Conv2d(32, 8×8, stride=4) → ReLU   → (32, 20, 20)
  → Conv2d(64, 4×4, stride=2) → ReLU   → (64,  9,  9)
  → Conv2d(64, 3×3, stride=1) → ReLU   → (64,  7,  7)
  → Flatten                             → (3136,)
  → Linear(3136, 512)         → ReLU
  → Linear(512, n_actions)              → Q(NOOP), Q(UP), Q(DOWN)
```

### Key Hyperparameters

| Parameter | Value |
|---|---|
| Learning rate | 0.0001 (Adam) |
| Discount γ | 0.99 |
| ε start / end | 1.0 → 0.05 |
| ε decay | 0.9995 per step |
| Batch size | 32 |
| Replay buffer | 100,000 |
| Frame stack | 4 |
| Frame size | 84×84 grayscale |
| Action repeat | 4 (MaxAndSkip) |
| Target net update | every 1,000 train steps |
| Loss | Huber (smooth-L1) |